# EmpowerLens — PatternReframe augmentation (flat multilabel)

Does adding ~8.7k borrowed "unhelpful thought" examples improve fine-grained
distortion typing? Flat multilabel only — **no cascade** — so it compares directly
against the existing baseline.

**Baseline to beat** (`mental-roberta-base`, flat multilabel, `data/splits`, test):

| | macro_f1 | weighted_f1 |
|---|---|---|
| flat baseline | **0.237 +/- 0.030** | 0.238 +/- 0.024 |
| cascade (reference) | 0.240 +/- 0.012 | 0.247 +/- 0.010 |

**Dataset.** PatternReframe (Maddela et al., ACL 2023) — 9,688 crowdsourced thoughts
written to exhibit a given pattern, persona-conditioned. 8,712 usable after mapping
9 of its 10 patterns onto our taxonomy.

### Three things that will shape the result

1. **No No-Distortion rows.** Every PatternReframe row is distorted. Merging into the
   full splits drops the negative class from **37% -> 7%** of train. Arm A will likely
   over-predict distortions because of that alone.
2. **`emotional_reasoning` gets zero coverage** — the key exists in `marked_patterns`
   but is 0 in all 9,688 rows. Nine classes get augmented, one does not.
3. **Median 17 words vs 129** in Annotated. Training on one-liners and testing on
   paragraphs is a real distribution shift, and the most likely reason this fails.

**Two arms.** Arm A is the flat task. Arm B stays distorted-only, which is what
PatternReframe structurally is, and avoids the class-imbalance confound. Run both —
the contrast between them is the finding.

**Evaluation never changes:** the untouched Annotated test set. Train augmented,
test natural.

In [ ]:
import os
os.chdir('/kaggle/working/')
!rm -rf /kaggle/working/empowerlens

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "nayab-space"

!git clone --branch $BRANCH $REPO_URL empowerlens
os.chdir('/kaggle/working/empowerlens')

!pip install --upgrade pip setuptools wheel
!pip install -q -r requirements-transformer.txt
!pip install -q sentencepiece protobuf

In [ ]:
# mental/mental-roberta-base is GATED. Accept its licence at
# https://huggingface.co/mental/mental-roberta-base while logged in, then attach the
# HF_TOKEN secret to THIS notebook: Add-ons -> Secrets -> tick HF_TOKEN.
# Secrets are per-notebook even though the token itself is account-level.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) - training will 401 until this is set.")

In [ ]:
# Pin to one GPU before any subprocess imports torch.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi -L

In [ ]:
# Reuse the cascade bootstrap: sh() with its pipe-drain fix, run_and_report,
# sync(), skip-if-already-done, hard timeouts. PARENT_SPLITS is not used here —
# this notebook passes its splits dirs explicitly.
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())


## 1 - Download PatternReframe and build the augmented splits

The `parl.ai` URL redirects twice; the line below uses the final one. ParlAI is not
required — it is a plain 2.4 MB tarball of JSONL.

In [ ]:
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import os, tarfile, urllib.request

URL = "https://dl.fbaipublicfiles.com/parlai/reframe_thoughts/reframe_thoughts_v0.1.tar.gz"
TAR = "/kaggle/working/reframe_thoughts.tar.gz"
SRC = "/kaggle/working/reframe_thoughts_dataset"

if not os.path.isdir(SRC):
    urllib.request.urlretrieve(URL, TAR)
    with tarfile.open(TAR) as t:
        t.extractall("/kaggle/working")
    if not os.path.isdir(SRC):          # some versions nest one level deeper
        for root, dirs, files in os.walk("/kaggle/working"):
            if "train.txt" in files and "reframe" in root:
                SRC = root
                break
print("source:", SRC, sorted(os.listdir(SRC)))

# Arm A - flat: merge into the FULL splits (keeps No-Distortion rows, but dilutes them)
!python -m src.make_splits_patternreframe --source $SRC --merge-into data/splits --out data/splits_pr --force

# Arm B - distorted-only: merge into Stage 2's splits (no negatives expected there)
!python -m src.make_splits_patternreframe --source $SRC --merge-into data/splits_stage2 --out data/splits_stage2_pr --force

In [ ]:
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import pandas as pd

# Sanity check BEFORE spending GPU time. Watch the No-Distortion share in Arm A.
for name, p in [("data/splits (base)", "data/splits"),
                ("data/splits_pr (Arm A)", "data/splits_pr"),
                ("data/splits_stage2 (base)", "data/splits_stage2"),
                ("data/splits_stage2_pr (Arm B)", "data/splits_stage2_pr")]:
    tr = pd.read_csv(f"{p}/train.csv", encoding="utf-8-sig")
    te = pd.read_csv(f"{p}/test.csv", encoding="utf-8-sig")
    nd = int((tr.y_bin == 0).sum())
    print(f"  {name:32} train={len(tr):6}  no-distortion={nd:5} ({100*nd/len(tr):4.1f}%)  test={len(te)}")

print("\nval/test must be IDENTICAL to their base - augmentation touches train only.")

## 2 - Arm A: flat multilabel on `data/splits_pr`

Full task, negative class present but diluted to ~7% of train.
**Compare against flat baseline macro_f1 0.237 +/- 0.030.**

In [ ]:
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

ARM_A_OUT = "results_pr_flat"
!mkdir -p $ARM_A_OUT

print("=== Arm A: flat multilabel, Annotated + PatternReframe ===")
for seed in SEEDS:
    run_and_report(
        "multilabel", "data/splits_pr", ARM_A_OUT, seed,
        ckpt_dir="/kaggle/working/checkpoints_pr_flat",
        extra_flags="--max-length 256 --truncation head_tail --batch-size 32",
    )
sync(ARM_A_OUT)

## 3 - Arm B: distorted-only on `data/splits_stage2_pr`

The structurally honest use of PatternReframe: every row distorted on both sides, so
class balance is not a confound. Train 1,278 -> 9,990.

Evaluated with `--allow-distorted-only`, so these are **isolated** numbers — comparable
only to Stage 2's isolated baseline (**macro_f1 0.262 +/- 0.023**), never to a flat
model or a cascade result.

In [ ]:
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

ARM_B_OUT = "results_pr_stage2"
!mkdir -p $ARM_B_OUT

print("=== Arm B: Stage-2 style, distorted-only, + PatternReframe ===")
for seed in SEEDS:
    run_and_report(
        "multilabel", "data/splits_stage2_pr", ARM_B_OUT, seed,
        ckpt_dir="/kaggle/working/checkpoints_pr_stage2",
        eval_flags="--allow-distorted-only",
        extra_flags=("--max-length 256 --truncation head_tail --batch-size 32 "
                     "--loss focal --focal-gamma 2.0 --llrd --llrd-decay 0.9 --lr 3e-5 "
                     "--lr-scheduler cosine --early-stopping-patience 2"),
    )
sync(ARM_B_OUT)

## 4 - Compare against the baselines

Arm A vs the flat baseline is the headline. Arm B vs Stage 2 isolated says whether
PatternReframe helps at all once class balance is removed as a confound.

In [ ]:
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import pandas as pd

SOURCES = {
    "flat baseline (Annotated only)":   "results_multilabel_flat",
    "Arm A: flat + PatternReframe":     "results_pr_flat",
    "Stage2 isolated (Annotated only)": "results_stage2",
    "Arm B: Stage2 + PatternReframe":   "results_pr_stage2",
}
frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p); d["run"] = label; frames.append(d)
    else:
        print(f"[skip] {p} not found")

if frames:
    allr = pd.concat(frames, ignore_index=True)
    v = allr[(allr.task == "multilabel") & (allr.split == "test")]
    print(v.groupby("run")[["weighted_f1", "macro_f1", "micro_f1"]].agg(["mean", "std"]).round(3).to_string())

## 5 - Zip for download

`/kaggle/working` is wiped when the session ends. Do this before closing the tab.

In [ ]:
BOOT = '/kaggle/working/empowerlens/notebooks/cascade_bootstrap.py'
exec(open(BOOT).read())

import os
ZIP = "/kaggle/working/results_patternreframe.zip"
folders = [f for f in ("results_pr_flat", "results_pr_stage2")
           if os.path.isdir(f"/kaggle/working/{f}") and os.listdir(f"/kaggle/working/{f}")]
if not folders:
    print("NOTHING TO ZIP.")
else:
    if os.path.exists(ZIP):
        os.remove(ZIP)
    sh("cd /kaggle/working && zip -rq " + os.path.basename(ZIP) + " " + " ".join(folders))
    print("\nWrote " + ZIP + " (%.1f MB)" % (os.path.getsize(ZIP)/1048576))
    print("Download from the Output panel, then locally:")
    print('   Expand-Archive -Path "$env:USERPROFILE\\Downloads\\results_patternreframe.zip" -DestinationPath . -Force')
    print("   venv\\Scripts\\python.exe -m src.compile_results")